# Train CCT-S in Jupyter

Use the bundled plate data or run the public dataset download notebook and set `USE_ENERGY_METER = True`. Each cell exposes the arrays, model, optimizer, loss, callbacks or predictions so you can inspect them individually. The CCT-S architecture and CSV dataset loader come from `fast-plate-ocr`.

In [ ]:
from pathlib import Path
import csv
import os
import random
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
USE_ENERGY_METER = os.environ.get("CCT_ENERGY_METER", "0") == "1"
DATASET = ROOT / ('dataset/energy-meter' if USE_ENERGY_METER else 'dummy_data')
PLATE_CONFIG = ROOT / ('config/cct_s_energy_meter_config.yaml' if USE_ENERGY_METER
                       else 'config/cct_s_v2_global_plate_config.yaml')
WORK = ROOT / os.environ.get("CCT_WORK", "artifacts/energy_meter" if USE_ENERGY_METER else "artifacts")
OUTPUT = ROOT / os.environ.get("CCT_OUTPUT", WORK if USE_ENERGY_METER else WORK / "training")
if USE_ENERGY_METER and not (DATASET / 'train_annotations.csv').is_file():
    raise FileNotFoundError('Run 00_download_ocr_example.ipynb first.')

EPOCHS = 1
BATCH_SIZE = 4
LEARNING_RATE = 5e-4
SEED = 20261114
INITIAL_WEIGHTS = None  # For example, ROOT / "pretrained/cct_s_v2_global.keras"
EPOCHS = 1
BATCH_SIZE = 4
LEARNING_RATE = 5e-4
SEED = 20261114
INITIAL_WEIGHTS = None  # For example, ROOT / "pretrained/cct_s_v2_global.keras"

## Inspect the data and make a training-only holdout

In [ ]:
from fast_plate_ocr.train.data.dataset import PlateRecognitionPyDataset
from fast_plate_ocr.train.data.augmentation import default_train_augmentation
from fast_plate_ocr.train.model.config import load_plate_config_from_yaml

plate_config = load_plate_config_from_yaml(PLATE_CONFIG)
with (DATASET / "train_annotations.csv").open(newline="") as stream:
    records = list(csv.DictReader(stream))
labels = sorted({row["plate_text"] for row in records})
if len(labels) < 2:
    raise ValueError("Need two distinct labels for a training holdout.")
random.Random(SEED).shuffle(labels)
held_labels = set(labels[:max(1, int(len(labels) * 0.15))])
fit = [row for row in records if row["plate_text"] not in held_labels]
holdout = [row for row in records if row["plate_text"] in held_labels]

RUN = OUTPUT / "notebook"
if (RUN / "best.keras").exists():
    raise FileExistsError("Choose a new OUTPUT to preserve the previous run.")
RUN.mkdir(parents=True, exist_ok=True)

def write_annotations(name, rows):
    path = RUN / f"{name}.csv"
    with path.open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        for row in rows:
            writer.writerow({**row, "image_path": os.path.relpath(DATASET / row["image_path"], start=RUN)})
    return path

augmentation = default_train_augmentation(img_color_mode=plate_config.image_color_mode)
train_data = PlateRecognitionPyDataset(write_annotations("fit", fit), plate_config=plate_config,
                                       batch_size=BATCH_SIZE, shuffle=True, transform=augmentation)
held_data = PlateRecognitionPyDataset(write_annotations("holdout", holdout), plate_config=plate_config,
                                      batch_size=BATCH_SIZE, shuffle=False)
with Image.open(DATASET / fit[0]["image_path"]) as image:
    display(image.copy())
batch_pixels, batch_labels = train_data[0]
print("Training / holdout:", len(fit), len(holdout), "Batch shape:", batch_pixels.shape)
print("Example label:", fit[0]["plate_text"])

## Build, compile, and train the Keras model

In [ ]:
import keras
from fast_plate_ocr.train.model.model_schema import load_model_config_from_yaml
from fast_plate_ocr.train.model.model_builders import build_model
from fast_plate_ocr.train.model.loss import cce_loss
from fast_plate_ocr.train.model.metric import plate_acc_metric

keras.utils.set_random_seed(SEED)
model_config = load_model_config_from_yaml(ROOT / "config/cct_s_v2_global_model_config.yaml")
model = build_model(model_config, plate_config, enable_region_head=False)
if INITIAL_WEIGHTS:
    model.load_weights(INITIAL_WEIGHTS, skip_mismatch=True)

optimizer = keras.optimizers.AdamW(learning_rate=LEARNING_RATE, weight_decay=5e-4)
loss = cce_loss(vocabulary_size=plate_config.vocabulary_size, label_smoothing=0)
metric = plate_acc_metric(max_plate_slots=plate_config.max_plate_slots,
                          vocabulary_size=plate_config.vocabulary_size)
model.compile(optimizer=optimizer, loss={"plate": loss}, metrics={"plate": [metric]}, jit_compile=False)
model.summary()

# Inspect model(batch_pixels, training=False) or change the callbacks before fitting.
best = RUN / "best.keras"
callbacks = [
    keras.callbacks.ModelCheckpoint(best, monitor=f"val_{metric.__name__}",
                                    mode="max", save_best_only=True),
    keras.callbacks.TerminateOnNaN(),
    keras.callbacks.CSVLogger(RUN / "training_log.csv"),
]
history = model.fit(train_data, epochs=EPOCHS, validation_data=held_data, callbacks=callbacks)
if not best.is_file():
    raise RuntimeError(f"Checkpoint metric was not logged. Available: {list(history.history)}")
print("Checkpoint:", best)

## Inspect full-string validation predictions

In [ ]:
import cv2
import numpy as np
from fast_plate_ocr.train.utilities.utils import load_keras_model

model = load_keras_model(best, plate_config)
with (DATASET / "val_annotations.csv").open(newline="") as stream:
    validation = list(csv.DictReader(stream))
predictions = []
for row in validation:
    with Image.open(DATASET / row["image_path"]) as image:
        rgb = np.asarray(image.convert("RGB"))
    pixels = cv2.resize(rgb, (128, 64), interpolation=cv2.INTER_LINEAR)[None].astype(np.float32)
    output = model(pixels, training=False)
    logits = output["plate"] if isinstance(output, dict) else output
    tokens = keras.ops.convert_to_numpy(logits)[0].argmax(-1)
    text = "".join(plate_config.alphabet[i] for i in tokens).split(plate_config.pad_char, 1)[0]
    predictions.append({**row, "prediction": text})
exact = sum(r["plate_text"] == r["prediction"] for r in predictions)
print(f"Validation exact match: {exact}/{len(predictions)}")
display(predictions[:10])